# Task 1: Data Cleaning and Preprocessing
### Project: Network Intrusion Detection Using Supervised Machine Learning
### Dataset: CICIDS2017 (12 Consolidated Classes)

In this notebook, we load the raw CICIDS2017 dataset from all 8 CSV files, clean infinite and missing values, consolidate labels into 12 distinct classes, and drop zero-variance constant features.

In [ ]:
# Step 1: Import foundational libraries
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries loaded successfully!")

In [ ]:
# Step 2: Load and combine all 8 CSV files
file_list = glob.glob('/content/MachineLearningCVE/*.csv')
df_list = []

for file in file_list:
    print("Reading file:", file)
    data = pd.read_csv(file)
    data.columns = data.columns.str.strip()
    
    # downcast float64 to float32 to save memory in Colab
    for col in data.select_dtypes(include=['float64']).columns:
        data[col] = data[col].astype('float32')
    for col in data.select_dtypes(include=['int64']).columns:
        data[col] = data[col].astype('int32')
        
    df_list.append(data)

df = pd.concat(df_list, ignore_index=True)
print("Combined dataset shape:", df.shape)

In [ ]:
# Step 3: Handle infinite values and missing values
df.replace([np.inf, -np.inf], np.nan, inplace=True)
before_clean = len(df)
df.dropna(inplace=True)
print("Dropped rows with NaN or infinite values:", before_clean - len(df))

In [ ]:
# Step 4: Consolidate labels into 12 standardized classes
# Merge Web Attacks
df.loc[df['Label'].str.contains('Web Attack', case=False, na=False), 'Label'] = 'Web Attack'

# Merge Heartbleed into Infiltration
df['Label'] = df['Label'].replace({'Heartbleed': 'Infiltration'})

print("Total unique classes:", df['Label'].nunique())
print("Class distribution:\n", df['Label'].value_counts())

In [ ]:
# Step 5: Remove constant features (zero-variance columns)
constant_cols = [col for col in df.columns if col != 'Label' and df[col].nunique() == 1]
print("Dropping constant columns:", constant_cols)
df.drop(columns=constant_cols, inplace=True)

# Check for data leakage columns
leakage_cols = ['Flow ID', 'Source IP', 'Destination IP', 'Timestamp']
found_leakage = [col for col in leakage_cols if col in df.columns]
if found_leakage:
    df.drop(columns=found_leakage, inplace=True)

print("Final clean shape:", df.shape)